# DOE Alaska MTE/IGS Production Potential Export

This notebook is for the DOE Anaconda / VS Code desktop. It reads the full Excel
workbooks from the same folder used by the ML pipeline notebook and exports an
Alaska-only Excel workbook for the two project wells:

- MTE / Mt. Elbert
- IGS / Ignik Sikumi

It excludes Canada/Mallik wells by default. If true production columns are found
in the Alaska rows, the notebook summarizes them. If the sheets only contain
well-log and saturation columns, it computes cumulative depth-integrated
production-potential proxies:

- gas/hydrate potential from porosity x hydrate saturation x interval thickness
- water potential from porosity x water saturation x interval thickness
- sand/reservoir-quality proxy from gamma ray, porosity, caliper, and interval thickness

Important: depth-integrated log proxies are not measured production history.


In [ ]:
# =============================================================================
# 1. User controls
# =============================================================================

from __future__ import annotations

from datetime import datetime
from pathlib import Path
import math
import os
import re
import warnings

import numpy as np
import pandas as pd


CODE_VERSION = "DOE_ALASKA_MTE_IGS_PRODUCTION_POTENTIAL_EXPORT_V1_2026_06_30"

# Default matches the other DOE notebook:
# docs/current_source_bundle_2026_06_26/DOE_MASTER_ML_PIPELINE_EQUATION_FIRST_V13_ALL_IN_ONE_CORE_PARALLEL_ANN.ipynb
INPUT_DIR = Path(
    os.environ.get(
        "NSGH_INPUT_DIR",
        str(Path.home() / "Downloads" / "Northslopedatasets06052026"),
    )
).expanduser()

OUTPUT_ROOT = Path(os.environ.get("NSGH_OUTPUT_DIR", str(INPUT_DIR / "outputs_runtime"))).expanduser()

# Leave blank to summarize the full well. Set both values to summarize one interval.
# Example in a first cell before running this notebook:
#   import os
#   os.environ["NSGH_INTERVAL_TOP_FT"] = "2100"
#   os.environ["NSGH_INTERVAL_BASE_FT"] = "2400"
SELECTED_INTERVAL_TOP_FT = os.environ.get("NSGH_INTERVAL_TOP_FT", "").strip()
SELECTED_INTERVAL_BASE_FT = os.environ.get("NSGH_INTERVAL_BASE_FT", "").strip()

SELECTED_INTERVAL_TOP_FT = float(SELECTED_INTERVAL_TOP_FT) if SELECTED_INTERVAL_TOP_FT else None
SELECTED_INTERVAL_BASE_FT = float(SELECTED_INTERVAL_BASE_FT) if SELECTED_INTERVAL_BASE_FT else None

# These thresholds are review settings, not final scientific cutoffs.
DEFAULT_UNLABELED_DEPTH_UNIT = os.environ.get("NSGH_DEFAULT_DEPTH_UNIT", "ft").strip().lower()
CLEAN_SAND_GR_API = float(os.environ.get("NSGH_CLEAN_SAND_GR_API", "80"))
CALIPER_REVIEW_IN = float(os.environ.get("NSGH_CALIPER_REVIEW_IN", "10"))
REFINED_TARGET_MAX_OFFSET_FT = float(os.environ.get("NSGH_REFINED_TARGET_MAX_OFFSET_FT", "5"))

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT / f"alaska_mte_igs_production_potential_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CODE_VERSION:", CODE_VERSION)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Selected interval ft:", SELECTED_INTERVAL_TOP_FT, SELECTED_INTERVAL_BASE_FT)


In [ ]:
# =============================================================================
# 2. Header maps and helper functions
# =============================================================================

PROJECT_WELLS = {
    "MTE": {
        "display_name": "Mt. Elbert / MTE",
        "tokens": (
            "mte",
            "mtelbert",
            "mountelbert",
            "mtelbert1",
            "mt elbert",
            "mount elbert",
            "wellmte",
            "wellc",
        ),
    },
    "IGS": {
        "display_name": "Ignik Sikumi / IGS",
        "tokens": (
            "igs",
            "ignik",
            "igniksikumi",
            "ignik sikumi",
            "sikumi",
            "welligs",
            "welld",
        ),
    },
}

EXCLUDED_NON_ALASKA_TOKENS = (
    "mallik",
    "2l38",
    "5l38",
    "canada",
    "mackenzie",
    "wella",
    "wellb",
)

COLUMN_ALIASES = {
    "depth": (
        "Depth_ft",
        "Depth (ft)",
        "Depth, ft",
        "DEPT",
        "DEPTH",
        "True Depth",
        "depth_m",
        "Depth_m",
        "MD",
    ),
    "porosity": (
        "phi_den",
        "DPHI",
        "density_porosity_vv",
        "Phi_porosity",
        "phi_nmr",
        "NMRPHI",
        "NPHI",
        "phi_neut",
        "neutron_porosity_vv",
    ),
    "porosity_primary": (
        "phi_den",
        "DPHI",
        "density_porosity_vv",
        "Phi_porosity",
    ),
    "porosity_fallback": (
        "phi_nmr",
        "NMRPHI",
        "NPHI",
        "phi_neut",
        "neutron_porosity_vv",
    ),
    "hydrate_saturation": (
        "S_h",
        "Sh",
        "Sgh",
        "S_g_h",
        "Hydrate Saturation",
        "hydrate_saturation",
        "hydrate_saturation_reference",
        "NMR_SAT",
    ),
    "water_saturation": (
        "S_wr",
        "Swr",
        "S_w",
        "Sw",
        "Water Saturation",
        "water_saturation",
        "irreducible_water_saturation_vv",
    ),
    "gamma_ray": ("GR", "gr_api", "Gamma Ray", "GammaRay"),
    "caliper": ("CAL1", "caliper", "Caliper", "CALI", "HCAL"),
    "resistivity": ("AO90", "A090", "AF90", "RES", "RT", "Rt", "Deep formation resistivity", "rt_ohm_m"),
    "vp": ("VELP", "VP", "Vp", "vp_m_s"),
    "vs": ("VS1", "VS", "Vs", "vs_m_s"),
    "well": ("WELL", "Well", "well_alias", "Well Name", "well_name", "source_well", "Dataset"),
    "date": ("Date", "ReportDate", "ProductionDate", "TestDate", "Time", "DateTime"),
    "gas_produced": ("GasProduced", "gas_produced", "gas produced", "cumulative_gas", "cum_gas", "gas_rate"),
    "water_produced": ("WaterProduced", "water_produced", "water produced", "cumulative_water", "cum_water", "water_rate"),
    "sand_produced": (
        "SandProduced",
        "sand_produced",
        "sand produced",
        "solids_produced",
        "produced_sand",
        "sand_mass",
    ),
}


def normalize_token(value: object) -> str:
    return "".join(ch for ch in str(value).strip().lower() if ch.isalnum())


def readable(value: object) -> str:
    return "" if value is None or (isinstance(value, float) and math.isnan(value)) else str(value)


def infer_project_well(*values: object) -> str | None:
    text = " ".join(readable(v).lower() for v in values if readable(v))
    normalized = normalize_token(text)
    if not normalized:
        return None
    for well, spec in PROJECT_WELLS.items():
        for token in spec["tokens"]:
            if normalize_token(token) in normalized:
                return well
    return None


def contains_excluded_non_alaska_text(*values: object) -> bool:
    normalized = normalize_token(" ".join(readable(v) for v in values))
    return any(token in normalized for token in EXCLUDED_NON_ALASKA_TOKENS)


def find_column(frame: pd.DataFrame, role: str) -> str | None:
    lookup = {normalize_token(column): column for column in frame.columns}
    for alias in COLUMN_ALIASES[role]:
        normalized = normalize_token(alias)
        if normalized in lookup:
            return lookup[normalized]
    for column in frame.columns:
        normalized_column = normalize_token(column)
        for alias in COLUMN_ALIASES[role]:
            normalized_alias = normalize_token(alias)
            if normalized_alias and normalized_alias in normalized_column:
                return column
    return None


def first_non_null_text(series: pd.Series, max_values: int = 8) -> str:
    values = series.dropna().astype(str).head(max_values).tolist()
    return " | ".join(values)


def numeric_series(frame: pd.DataFrame, column: str | None) -> pd.Series:
    if column is None or column not in frame:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce")


def normalize_fraction(values: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce")
    finite = numeric[np.isfinite(numeric)]
    if finite.empty:
        return numeric
    if finite.quantile(0.95) > 1.5:
        numeric = numeric / 100.0
    return numeric


def calculation_fraction(values: pd.Series) -> pd.Series:
    return normalize_fraction(values).clip(lower=0.0, upper=1.0)


def depth_to_ft(values: pd.Series, column_name: str | None) -> tuple[pd.Series, str]:
    numeric = pd.to_numeric(values, errors="coerce")
    normalized = normalize_token(column_name or "")
    if "depthm" in normalized or normalized.endswith("m") or "(m)" in str(column_name).lower():
        return numeric * 3.280839895, "m_to_ft"
    if DEFAULT_UNLABELED_DEPTH_UNIT in {"m", "meter", "meters"} and "ft" not in normalized:
        return numeric * 3.280839895, "assumed_m_to_ft"
    return numeric, "assumed_ft"


def safe_sheet_name(name: str) -> str:
    cleaned = re.sub(r"[\[\]\:\*\?\/\\]", "_", name).strip()
    return (cleaned or "Sheet")[:31]


def display_frame(frame: pd.DataFrame, rows: int = 10) -> None:
    try:
        display(frame.head(rows))
    except NameError:
        print(frame.head(rows).to_string(index=False))


In [ ]:
# =============================================================================
# 3. Discover Alaska workbook sheets
# =============================================================================

def list_excel_files(input_dir: Path) -> list[Path]:
    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder does not exist: {input_dir}")
    files = []
    for path in sorted(input_dir.glob("*.xls*")):
        lower = path.name.lower()
        if path.name.startswith("~$"):
            continue
        if "alaska_mte_igs_production_potential" in lower:
            continue
        if lower.endswith((".xlsx", ".xlsm", ".xls")):
            files.append(path)
    return files


def sample_sheet(path: Path, sheet_name: str, rows: int = 50) -> pd.DataFrame:
    try:
        return pd.read_excel(path, sheet_name=sheet_name, nrows=rows)
    except Exception as exc:
        warnings.warn(f"Could not sample {path.name}::{sheet_name}: {exc}")
        return pd.DataFrame()


def full_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
    return pd.read_excel(path, sheet_name=sheet_name)


def discover_alaska_sources(input_dir: Path) -> tuple[list[dict[str, object]], pd.DataFrame]:
    selected: list[dict[str, object]] = []
    scan_rows: list[dict[str, object]] = []
    for path in list_excel_files(input_dir):
        try:
            with pd.ExcelFile(path) as excel:
                sheet_names = list(excel.sheet_names)
        except Exception as exc:
            scan_rows.append({
                "file_name": path.name,
                "sheet_name": "",
                "status": "open_failed",
                "reason": str(exc),
            })
            continue
        for sheet_name in sheet_names:
            header = sample_sheet(path, sheet_name, rows=50)
            columns = list(header.columns)
            text_context = f"{path.name} {sheet_name} {' '.join(map(str, columns))}"
            well_from_context = infer_project_well(path.name, sheet_name)
            well_from_rows = None
            well_col = find_column(header, "well") if not header.empty else None
            if well_col is not None:
                for value in header[well_col].dropna().astype(str).unique()[:20]:
                    inferred = infer_project_well(value)
                    if inferred:
                        well_from_rows = inferred
                        break
            well = well_from_context or well_from_rows
            excluded = contains_excluded_non_alaska_text(path.name, sheet_name)
            has_known_alaska_header = any(find_column(header, role) is not None for role in ("depth", "porosity", "hydrate_saturation", "water_saturation"))
            selected_status = "selected" if well else "skipped"
            reason = "project_well_detected" if well else "no_mte_or_igs_alias_in_file_sheet_or_sample_rows"
            if excluded and not well:
                reason = "non_alaska_canada_mallik_alias"
            scan_rows.append({
                "file_name": path.name,
                "sheet_name": sheet_name,
                "status": selected_status,
                "reason": reason,
                "detected_well": well or "",
                "sample_rows": len(header),
                "column_count": len(columns),
                "has_known_alaska_header": bool(has_known_alaska_header),
                "columns_preview": " | ".join(map(str, columns[:25])),
            })
            if well:
                frame = full_sheet(path, sheet_name)
                selected.append({
                    "file_path": path,
                    "file_name": path.name,
                    "sheet_name": sheet_name,
                    "well_alias": well,
                    "frame": frame,
                })
    return selected, pd.DataFrame(scan_rows)


selected_sources, input_scan_df = discover_alaska_sources(INPUT_DIR)
print(f"Selected Alaska source sheets: {len(selected_sources)}")
display_frame(input_scan_df, rows=30)


In [ ]:
# =============================================================================
# 4. Build interval-level production-potential tables
# =============================================================================

def split_frame_to_project_wells(source: dict[str, object]) -> list[dict[str, object]]:
    frame = source["frame"].copy()
    well_col = find_column(frame, "well")
    if well_col is None:
        frame["well_alias"] = source["well_alias"]
        return [{**source, "well_alias": source["well_alias"], "frame": frame}]

    frames = []
    detected = frame[well_col].map(infer_project_well)
    for well in PROJECT_WELLS:
        subset = frame[detected.eq(well)].copy()
        if len(subset):
            subset["well_alias"] = well
            frames.append({**source, "well_alias": well, "frame": subset})
    if not frames:
        frame["well_alias"] = source["well_alias"]
        frames.append({**source, "well_alias": source["well_alias"], "frame": frame})
    return frames


def build_header_mapping_rows(source: dict[str, object]) -> list[dict[str, object]]:
    frame = source["frame"]
    rows = []
    for role in [
        "depth",
        "porosity_primary",
        "porosity_fallback",
        "hydrate_saturation",
        "water_saturation",
        "gamma_ray",
        "caliper",
        "resistivity",
        "vp",
        "vs",
        "gas_produced",
        "water_produced",
        "sand_produced",
    ]:
        column = find_column(frame, role)
        rows.append({
            "file_name": source["file_name"],
            "sheet_name": source["sheet_name"],
            "well_alias": source["well_alias"],
            "role": role,
            "matched_header": column or "",
            "status": "found" if column else "missing",
        })
    return rows


def compute_interval_thickness_ft(depth_ft: pd.Series) -> pd.Series:
    diffs = depth_ft.shift(-1) - depth_ft
    positive = diffs[diffs > 0]
    fallback = float(positive.median()) if len(positive) else np.nan
    thickness = diffs.where(diffs > 0, fallback)
    return thickness.clip(lower=0)


def clean_sand_score_from_gr(gr: pd.Series) -> pd.Series:
    # 1 is cleaner/lower GR, 0 is shalier/higher GR under a simple review threshold.
    score = (CLEAN_SAND_GR_API - pd.to_numeric(gr, errors="coerce")) / CLEAN_SAND_GR_API
    return score.clip(lower=0.0, upper=1.0)


def build_interval_potential(source: dict[str, object]) -> pd.DataFrame:
    frame = source["frame"].copy()
    depth_col = find_column(frame, "depth")
    if depth_col is None:
        return pd.DataFrame()

    depth_ft, depth_unit_policy = depth_to_ft(numeric_series(frame, depth_col), depth_col)
    source_well = str(source["well_alias"])

    primary_porosity_col = find_column(frame, "porosity_primary")
    fallback_porosity_col = find_column(frame, "porosity_fallback")
    hydrate_col = find_column(frame, "hydrate_saturation")
    water_col = find_column(frame, "water_saturation")
    gr_col = find_column(frame, "gamma_ray")
    caliper_col = find_column(frame, "caliper")
    resistivity_col = find_column(frame, "resistivity")
    vp_col = find_column(frame, "vp")
    vs_col = find_column(frame, "vs")

    primary_porosity = calculation_fraction(numeric_series(frame, primary_porosity_col))
    fallback_porosity = calculation_fraction(numeric_series(frame, fallback_porosity_col))
    porosity = primary_porosity.where(primary_porosity.notna(), fallback_porosity)
    porosity_source = np.where(primary_porosity.notna(), primary_porosity_col or "", fallback_porosity_col or "")

    hydrate = calculation_fraction(numeric_series(frame, hydrate_col))
    water = calculation_fraction(numeric_series(frame, water_col))
    gr = numeric_series(frame, gr_col)
    caliper = numeric_series(frame, caliper_col)
    resistivity = numeric_series(frame, resistivity_col)
    vp = numeric_series(frame, vp_col)
    vs = numeric_series(frame, vs_col)

    out = pd.DataFrame(
        {
            "well_alias": source_well,
            "well_name": PROJECT_WELLS[source_well]["display_name"],
            "source_file": source["file_name"],
            "source_sheet": source["sheet_name"],
            "source_row_index": frame.index,
            "depth_ft": depth_ft,
            "depth_unit_policy": depth_unit_policy,
            "porosity_used_vv": porosity,
            "porosity_source_header": porosity_source,
            "hydrate_saturation_used_vv": hydrate,
            "hydrate_saturation_source_header": hydrate_col or "",
            "water_saturation_used_vv": water,
            "water_saturation_source_header": water_col or "",
            "gr_api": gr,
            "gr_source_header": gr_col or "",
            "caliper_in": caliper,
            "caliper_source_header": caliper_col or "",
            "resistivity_ohm_m": resistivity,
            "resistivity_source_header": resistivity_col or "",
            "vp_value": vp,
            "vp_source_header": vp_col or "",
            "vs_value": vs,
            "vs_source_header": vs_col or "",
        }
    )
    out = out.dropna(subset=["depth_ft"]).sort_values("depth_ft").reset_index(drop=True)
    if out.empty:
        return out

    out["interval_thickness_ft"] = compute_interval_thickness_ft(out["depth_ft"])
    out["interval_base_ft"] = out["depth_ft"] + out["interval_thickness_ft"]

    out["gas_hydrate_pore_volume_index_ft"] = (
        out["interval_thickness_ft"] * out["porosity_used_vv"] * out["hydrate_saturation_used_vv"]
    )
    out["water_pore_volume_index_ft"] = (
        out["interval_thickness_ft"] * out["porosity_used_vv"] * out["water_saturation_used_vv"]
    )
    mobile_water_fraction = (1.0 - out["hydrate_saturation_used_vv"] - out["water_saturation_used_vv"]).clip(lower=0.0)
    out["mobile_water_proxy_index_ft"] = out["interval_thickness_ft"] * out["porosity_used_vv"] * mobile_water_fraction

    out["clean_sand_score"] = clean_sand_score_from_gr(out["gr_api"])
    out["clean_sand_flag_review"] = np.where(out["gr_api"].notna(), out["gr_api"] <= CLEAN_SAND_GR_API, pd.NA)
    out["caliper_review_flag"] = np.where(out["caliper_in"].notna(), out["caliper_in"] > CALIPER_REVIEW_IN, pd.NA)
    out["sand_reservoir_quality_index_ft"] = (
        out["interval_thickness_ft"] * out["porosity_used_vv"] * out["clean_sand_score"]
    )
    out["sand_produced_status"] = "not_measured_from_log_headers_proxy_only"
    out["production_label"] = "depth_integrated_potential_not_measured_production"
    out["cumulative_gas_hydrate_pore_volume_index_ft"] = out["gas_hydrate_pore_volume_index_ft"].fillna(0).cumsum()
    out["cumulative_water_pore_volume_index_ft"] = out["water_pore_volume_index_ft"].fillna(0).cumsum()
    out["cumulative_mobile_water_proxy_index_ft"] = out["mobile_water_proxy_index_ft"].fillna(0).cumsum()
    out["cumulative_sand_reservoir_quality_index_ft"] = out["sand_reservoir_quality_index_ft"].fillna(0).cumsum()
    return out


expanded_sources: list[dict[str, object]] = []
for source in selected_sources:
    expanded_sources.extend(split_frame_to_project_wells(source))

header_mapping_df = pd.DataFrame(
    [row for source in expanded_sources for row in build_header_mapping_rows(source)]
)

interval_frames = [build_interval_potential(source) for source in expanded_sources]
interval_potential_df = (
    pd.concat([frame for frame in interval_frames if not frame.empty], ignore_index=True)
    if any(not frame.empty for frame in interval_frames)
    else pd.DataFrame()
)

print("Interval rows:", len(interval_potential_df))
display_frame(header_mapping_df, rows=40)
display_frame(interval_potential_df, rows=10)


In [ ]:
# =============================================================================
# 5. Summaries: cumulative by well, selected interval, measured production scan
# =============================================================================

def selected_interval_filter(frame: pd.DataFrame) -> pd.Series:
    if frame.empty:
        return pd.Series([], dtype=bool)
    mask = pd.Series(True, index=frame.index)
    if SELECTED_INTERVAL_TOP_FT is not None:
        mask &= frame["depth_ft"] >= SELECTED_INTERVAL_TOP_FT
    if SELECTED_INTERVAL_BASE_FT is not None:
        mask &= frame["depth_ft"] <= SELECTED_INTERVAL_BASE_FT
    return mask


def summarize_interval_group(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    rows = []
    for well, group in frame.groupby("well_alias", dropna=False):
        rows.append({
            "summary_scope": label,
            "well_alias": well,
            "well_name": PROJECT_WELLS.get(str(well), {}).get("display_name", str(well)),
            "source_sheets": " | ".join(sorted(group["source_sheet"].dropna().astype(str).unique())),
            "row_count": int(len(group)),
            "depth_min_ft": float(group["depth_ft"].min()),
            "depth_max_ft": float(group["depth_ft"].max()),
            "total_interval_thickness_ft": float(group["interval_thickness_ft"].sum(skipna=True)),
            "cumulative_gas_hydrate_pore_volume_index_ft": float(group["gas_hydrate_pore_volume_index_ft"].sum(skipna=True)),
            "cumulative_water_pore_volume_index_ft": float(group["water_pore_volume_index_ft"].sum(skipna=True)),
            "cumulative_mobile_water_proxy_index_ft": float(group["mobile_water_proxy_index_ft"].sum(skipna=True)),
            "cumulative_sand_reservoir_quality_index_ft": float(group["sand_reservoir_quality_index_ft"].sum(skipna=True)),
            "mean_porosity_used_vv": float(group["porosity_used_vv"].mean(skipna=True)),
            "mean_hydrate_saturation_used_vv": float(group["hydrate_saturation_used_vv"].mean(skipna=True)),
            "mean_water_saturation_used_vv": float(group["water_saturation_used_vv"].mean(skipna=True)),
            "clean_sand_flagged_rows": int(pd.Series(group["clean_sand_flag_review"]).fillna(False).sum()),
            "caliper_review_flagged_rows": int(pd.Series(group["caliper_review_flag"]).fillna(False).sum()),
            "production_label": "depth_integrated_potential_not_measured_production",
        })
    return pd.DataFrame(rows)


def scan_measured_production(source: dict[str, object]) -> pd.DataFrame:
    frame = source["frame"].copy()
    gas_col = find_column(frame, "gas_produced")
    water_col = find_column(frame, "water_produced")
    sand_col = find_column(frame, "sand_produced")
    if gas_col is None and water_col is None and sand_col is None:
        return pd.DataFrame()
    date_col = find_column(frame, "date")
    out = pd.DataFrame({
        "well_alias": source["well_alias"],
        "well_name": PROJECT_WELLS[str(source["well_alias"])]["display_name"],
        "source_file": source["file_name"],
        "source_sheet": source["sheet_name"],
        "source_row_index": frame.index,
        "production_date_or_interval": frame[date_col] if date_col else "",
        "gas_produced_measured": numeric_series(frame, gas_col),
        "water_produced_measured": numeric_series(frame, water_col),
        "sand_produced_measured": numeric_series(frame, sand_col),
        "gas_source_header": gas_col or "",
        "water_source_header": water_col or "",
        "sand_source_header": sand_col or "",
        "production_label": "measured_or_source_production_column_found",
    })
    return out


cumulative_by_well_df = summarize_interval_group(interval_potential_df, "full_detected_depth_range")

selected_interval_df = interval_potential_df[selected_interval_filter(interval_potential_df)].copy()
selected_interval_summary_df = summarize_interval_group(
    selected_interval_df,
    (
        f"selected_interval_{SELECTED_INTERVAL_TOP_FT}_to_{SELECTED_INTERVAL_BASE_FT}_ft"
        if SELECTED_INTERVAL_TOP_FT is not None or SELECTED_INTERVAL_BASE_FT is not None
        else "full_detected_depth_range_no_selected_interval_set"
    ),
)

measured_frames = [scan_measured_production(source) for source in expanded_sources]
measured_production_df = (
    pd.concat([frame for frame in measured_frames if not frame.empty], ignore_index=True)
    if any(not frame.empty for frame in measured_frames)
    else pd.DataFrame(
        [
            {
                "status": "no_measured_production_columns_found",
                "note": (
                    "The detected MTE/IGS sheets did not include gas produced, water produced, "
                    "or sand produced columns. Use the potential/proxy sheets for depth-integrated "
                    "producibility, not measured production history."
                ),
            }
        ]
    )
)

if {"gas_produced_measured", "water_produced_measured", "sand_produced_measured"}.issubset(measured_production_df.columns):
    measured_cumulative_df = (
        measured_production_df.groupby(["well_alias", "well_name"], dropna=False)[
            ["gas_produced_measured", "water_produced_measured", "sand_produced_measured"]
        ]
        .sum(min_count=1)
        .reset_index()
    )
else:
    measured_cumulative_df = pd.DataFrame(
        [
            {
                "status": "not_available_from_detected_columns",
                "note": "Measured production summary is empty because no measured production columns were found.",
            }
        ]
    )

missing_production_data_df = pd.DataFrame(
    [
        {
            "desired_output": "gas produced",
            "status_from_known_mte_igs_headers": "not directly measured",
            "what_this_notebook_outputs": "gas_hydrate_pore_volume_index_ft",
            "required_for_true_production": "gas rate or cumulative gas, date/time or test interval, volume basis",
        },
        {
            "desired_output": "water produced",
            "status_from_known_mte_igs_headers": "not directly measured",
            "what_this_notebook_outputs": "water_pore_volume_index_ft and mobile_water_proxy_index_ft",
            "required_for_true_production": "water rate or cumulative water, date/time or test interval",
        },
        {
            "desired_output": "sand produced",
            "status_from_known_mte_igs_headers": "not directly measured",
            "what_this_notebook_outputs": "sand_reservoir_quality_index_ft and clean_sand/caliper flags",
            "required_for_true_production": "produced sand or solids mass/volume/concentration",
        },
    ]
)

display_frame(cumulative_by_well_df)
display_frame(selected_interval_summary_df)
display_frame(measured_production_df)


In [ ]:
# =============================================================================
# 6. Export the Excel workbook
# =============================================================================

def make_readme_frame() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"field": "code_version", "value": CODE_VERSION},
            {"field": "generated_at", "value": datetime.now().isoformat(timespec="seconds")},
            {"field": "input_folder", "value": str(INPUT_DIR)},
            {"field": "output_folder", "value": str(OUTPUT_DIR)},
            {"field": "included_wells", "value": "MTE / Mt. Elbert; IGS / Ignik Sikumi"},
            {"field": "excluded_wells", "value": "Mallik / Canada / WellA / WellB aliases are not included."},
            {
                "field": "gas_output",
                "value": (
                    "gas_hydrate_pore_volume_index_ft = interval_thickness_ft * porosity * hydrate_saturation. "
                    "This is a depth-integrated potential index, not measured production unless a measured gas column is found."
                ),
            },
            {
                "field": "water_output",
                "value": (
                    "water_pore_volume_index_ft = interval_thickness_ft * porosity * water_saturation. "
                    "mobile_water_proxy_index_ft uses max(0, 1 - hydrate_saturation - water_saturation)."
                ),
            },
            {
                "field": "sand_output",
                "value": (
                    "sand_reservoir_quality_index_ft is a clean-sand/reservoir-quality proxy from GR, porosity, "
                    "and interval thickness. It is not measured produced sand."
                ),
            },
            {"field": "selected_interval_top_ft", "value": SELECTED_INTERVAL_TOP_FT},
            {"field": "selected_interval_base_ft", "value": SELECTED_INTERVAL_BASE_FT},
            {"field": "clean_sand_gr_api_review_threshold", "value": CLEAN_SAND_GR_API},
            {"field": "caliper_review_in_threshold", "value": CALIPER_REVIEW_IN},
        ]
    )


def format_workbook(path: Path) -> None:
    try:
        from openpyxl import load_workbook
        from openpyxl.styles import Alignment, Font, PatternFill
    except Exception as exc:
        print(f"Workbook saved, but formatting skipped because openpyxl formatting import failed: {exc}")
        return
    workbook = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        for column_cells in worksheet.columns:
            values = [readable(cell.value) for cell in column_cells[:200]]
            max_len = max([len(value) for value in values] + [8])
            width = min(max(max_len + 2, 10), 46)
            worksheet.column_dimensions[column_cells[0].column_letter].width = width
        for row in worksheet.iter_rows():
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
    workbook.save(path)


OUTPUT_WORKBOOK = OUTPUT_DIR / f"alaska_mte_igs_production_potential_{RUN_ID}.xlsx"

sheets = {
    "README": make_readme_frame(),
    "Input_Scan": input_scan_df,
    "Header_Mapping": header_mapping_df,
    "Interval_Potential": interval_potential_df,
    "Cumulative_By_Well": cumulative_by_well_df,
    "Selected_Interval": selected_interval_summary_df,
    "Measured_Production": measured_production_df,
    "Measured_Cumulative": measured_cumulative_df,
    "Water_Risk_Proxy": interval_potential_df[
        [
            col
            for col in [
                "well_alias",
                "well_name",
                "source_sheet",
                "depth_ft",
                "interval_thickness_ft",
                "porosity_used_vv",
                "hydrate_saturation_used_vv",
                "water_saturation_used_vv",
                "water_pore_volume_index_ft",
                "mobile_water_proxy_index_ft",
                "cumulative_water_pore_volume_index_ft",
                "cumulative_mobile_water_proxy_index_ft",
            ]
            if col in interval_potential_df.columns
        ]
    ]
    if not interval_potential_df.empty
    else pd.DataFrame(),
    "Sand_Risk_Proxy": interval_potential_df[
        [
            col
            for col in [
                "well_alias",
                "well_name",
                "source_sheet",
                "depth_ft",
                "interval_thickness_ft",
                "gr_api",
                "clean_sand_score",
                "clean_sand_flag_review",
                "caliper_in",
                "caliper_review_flag",
                "sand_reservoir_quality_index_ft",
                "cumulative_sand_reservoir_quality_index_ft",
                "sand_produced_status",
            ]
            if col in interval_potential_df.columns
        ]
    ]
    if not interval_potential_df.empty
    else pd.DataFrame(),
    "Missing_Production_Data": missing_production_data_df,
}

with pd.ExcelWriter(OUTPUT_WORKBOOK, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        safe_name = safe_sheet_name(sheet_name)
        if frame is None or frame.empty:
            pd.DataFrame([{"status": "empty", "note": "No rows available for this sheet."}]).to_excel(
                writer,
                sheet_name=safe_name,
                index=False,
            )
        else:
            frame.to_excel(writer, sheet_name=safe_name, index=False)

format_workbook(OUTPUT_WORKBOOK)

print("Exported workbook:")
print(OUTPUT_WORKBOOK)
